# 02 — Training and Evaluation (Reusing Project Scripts)

Este notebook orquestra **treinamento, avaliação e evidências** (métricas, matriz de confusão, curvas e Grad-CAM) **reaproveitando os scripts do projeto** para evitar duplicação.

## Objetivos
- Executar o pipeline via `scripts/train.py` e `scripts/evaluate.py`
- Exibir artefatos gerados em `reports/`
- (Opcional) Comparar execuções via `scripts/compare_runs.py`
- (Opcional) Rodar inferência e Grad-CAM via CLI / outputs do app

> Observação: ajuste os argumentos conforme a interface real dos seus scripts.


In [1]:
from pathlib import Path
import sys
import subprocess
from datetime import datetime

PROJECT_ROOT = Path('..').resolve()
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'

PREFERRED_PROCESSED_NAME = 'hbfmid_cls_largest'

def select_processed_dataset_dir() -> Path:
    preferred = DATA_PROCESSED_ROOT / PREFERRED_PROCESSED_NAME
    if (preferred / 'manifest.csv').exists():
        return preferred
    candidates = []
    if DATA_PROCESSED_ROOT.exists():
        for d in DATA_PROCESSED_ROOT.iterdir():
            if d.is_dir() and (d / 'manifest.csv').exists():
                candidates.append(d)
    return sorted(candidates)[0] if candidates else preferred

DATA_PROCESSED_DIR = select_processed_dataset_dir()

for p in [DATA_RAW_DIR, DATA_PROCESSED_ROOT, MODELS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_PROCESSED_DIR:', DATA_PROCESSED_DIR)
print('MODELS_DIR:', MODELS_DIR)
print('REPORTS_DIR:', REPORTS_DIR)

def run_script(script_rel_path: str, args: list) -> int:
    script_path = PROJECT_ROOT / script_rel_path
    cmd = [sys.executable, str(script_path)] + args
    print('\n$ ' + ' '.join(cmd))
    return subprocess.call(cmd)


PROJECT_ROOT: D:\workspace-python\xray-bone-fracture-classifier
DATA_PROCESSED_DIR: D:\workspace-python\xray-bone-fracture-classifier\data\processed\hbfmid_cls_largest
MODELS_DIR: D:\workspace-python\xray-bone-fracture-classifier\models
REPORTS_DIR: D:\workspace-python\xray-bone-fracture-classifier\reports


## Runner utilitário para scripts

Função central para executar scripts via `subprocess`, capturar saída e padronizar logs.


In [2]:
def run_script(script_rel_path: str, args: list[str]) -> None:
    """Executa um script do projeto, imprimindo comando, stdout e stderr em caso de erro."""
    script_path = PROJECT_ROOT / script_rel_path
    if not script_path.exists():
        raise FileNotFoundError(f"Script não encontrado: {script_path}")

    cmd = [sys.executable, str(script_path)] + [str(a) for a in args]
    print("\n$ " + " ".join(cmd))

    result = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )

    if result.stdout:
        print(result.stdout)

    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"Falha ao executar: {script_rel_path} (exit={result.returncode})")


## (Opcional) Preparação do dataset para classificação

Se você já possui `data/processed` pronto, pode pular esta etapa.

Caso deseje que este notebook seja **Run All do zero**, mantenha este bloco e garanta que o dataset bruto esteja em `data/raw/`.


In [3]:
required_splits = [DATA_PROCESSED_DIR/'train', DATA_PROCESSED_DIR/'valid', DATA_PROCESSED_DIR/'test']
missing = [p for p in required_splits if not p.exists()]
print('Split dirs:')
for p in required_splits:
    print(' -', p, '->', 'OK' if p.exists() else 'MISSING')
if missing:
    raise FileNotFoundError('Dataset processado incompleto. Faltam: ' + ', '.join(str(p) for p in missing))


Split dirs:
 - D:\workspace-python\xray-bone-fracture-classifier\data\processed\hbfmid_cls_largest\train -> OK
 - D:\workspace-python\xray-bone-fracture-classifier\data\processed\hbfmid_cls_largest\valid -> OK
 - D:\workspace-python\xray-bone-fracture-classifier\data\processed\hbfmid_cls_largest\test -> OK


## Configuração do experimento

Registre aqui o racional das escolhas:
- Arquitetura / backbone (ex.: ResNet)
- Hiperparâmetros principais (lr, batch size, epochs)
- Estratégias de regularização (data augmentation, early stopping)
- Métrica(s) alvo (ex.: F1 macro)

Abaixo, mantenha os argumentos explicitamente no notebook para reprodutibilidade.


In [ ]:
TRAIN_PROFILE = 'final'  # 'baseline' | 'final'
PREFIX = 'run'           # mesmo valor do --prefix no CLI
DEVICE = 'directml'      # 'cpu' | 'cuda' | 'directml'
BATCH_SIZE = 16
NUM_WORKERS = 8

common = [
    '--data-dir', str(DATA_PROCESSED_DIR),
    '--models-dir', str(MODELS_DIR),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--seed', '42',
    '--prefix', PREFIX,
    '--device', DEVICE,
]

if TRAIN_PROFILE == 'baseline':
    train_args = common + ['--arch','baseline','--epochs','10','--lr','1e-3']
elif TRAIN_PROFILE == 'final':
    train_args = common + [
        '--arch','resnet18','--img-size','224','--epochs','15','--lr','1e-4',
        '--weight-decay','1e-4','--label-smoothing','0.0','--early-stopping-patience','5'
    ]
else:
    raise ValueError(f'TRAIN_PROFILE inválido: {TRAIN_PROFILE}')

print('Train args:', train_args)


Train args: ['--data-dir', 'D:\\workspace-python\\xray-bone-fracture-classifier\\data\\processed\\hbfmid_cls_largest', '--models-dir', 'D:\\workspace-python\\xray-bone-fracture-classifier\\models', '--batch-size', '24', '--num-workers', '6', '--seed', '42', '--prefix', 'run', '--device', 'directml', '--arch', 'resnet18', '--img-size', '224', '--epochs', '15', '--lr', '1e-4', '--weight-decay', '1e-4', '--label-smoothing', '0.0', '--early-stopping-patience', '5']


## Treinamento

Executa o treino via `scripts/train.py` e salva artefatos (pesos/modelos) em `runs/` (argumento `--models-dir`).


In [5]:
rc = run_script('scripts/train.py', train_args)
if rc != 0:
    raise RuntimeError(f'Treinamento falhou (rc={rc}).')
print('Treinamento concluído.')



$ d:\workspace-python\xray-bone-fracture-classifier\venv\Scripts\python.exe D:\workspace-python\xray-bone-fracture-classifier\scripts\train.py --data-dir D:\workspace-python\xray-bone-fracture-classifier\data\processed\hbfmid_cls_largest --models-dir D:\workspace-python\xray-bone-fracture-classifier\models --batch-size 24 --num-workers 6 --seed 42 --prefix run --device directml --arch resnet18 --img-size 224 --epochs 15 --lr 1e-4 --weight-decay 1e-4 --label-smoothing 0.0 --early-stopping-patience 5
OK. Run saved to: D:\workspace-python\xray-bone-fracture-classifier\models\run_20260114-101948



RuntimeError: Treinamento falhou (rc=None).

In [6]:
from pathlib import Path

def resolve_model_dir(models_dir: Path, prefix: str) -> Path:
    candidates = sorted(
        [p for p in models_dir.glob(f'{prefix}*') if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"Nenhum diretório em {models_dir} iniciando com '{prefix}'.")
    return candidates[0]

MODEL_DIR = resolve_model_dir(MODELS_DIR, PREFIX)
print('MODEL_DIR:', MODEL_DIR)


MODEL_DIR: D:\workspace-python\xray-bone-fracture-classifier\models\run_20260114-101948


## Avaliação

Executa a avaliação via `scripts/evaluate.py` e salva métricas/figuras em `reports/` (argumento `--reports-dir`).

**Atenção:** `--model-dir` deve apontar para o diretório do modelo/pesos gerados no treino.


In [7]:
REPORTS_RUN_DIR = REPORTS_DIR / MODEL_DIR.name
REPORTS_RUN_DIR.mkdir(parents=True, exist_ok=True)

eval_args = [
    '--data-dir', str(DATA_PROCESSED_DIR),
    '--model-dir', str(MODEL_DIR),
    '--batch-size', '32',
    '--num-workers', str(NUM_WORKERS),
    '--reports-dir', str(REPORTS_RUN_DIR),
]
print('Eval args:', eval_args)


Eval args: ['--data-dir', 'D:\\workspace-python\\xray-bone-fracture-classifier\\data\\processed\\hbfmid_cls_largest', '--model-dir', 'D:\\workspace-python\\xray-bone-fracture-classifier\\models\\run_20260114-101948', '--batch-size', '32', '--num-workers', '6', '--reports-dir', 'D:\\workspace-python\\xray-bone-fracture-classifier\\reports\\run_20260114-101948']


In [8]:
rc = run_script('scripts/evaluate.py', eval_args)
if rc != 0:
    raise RuntimeError(f'Avaliação falhou (rc={rc}).')
print('Avaliação concluída. Artefatos em:', REPORTS_RUN_DIR)



$ d:\workspace-python\xray-bone-fracture-classifier\venv\Scripts\python.exe D:\workspace-python\xray-bone-fracture-classifier\scripts\evaluate.py --data-dir D:\workspace-python\xray-bone-fracture-classifier\data\processed\hbfmid_cls_largest --model-dir D:\workspace-python\xray-bone-fracture-classifier\models\run_20260114-101948 --batch-size 32 --num-workers 6 --reports-dir D:\workspace-python\xray-bone-fracture-classifier\reports\run_20260114-101948
OK. Reports saved to: D:\workspace-python\xray-bone-fracture-classifier\reports\run_20260114-101948\run_20260114-101948



RuntimeError: Avaliação falhou (rc=None).

In [ ]:
# Predição (opcional)
# example_image = DATA_PROCESSED_DIR / 'test' / '<classe>' / 'example.png'
# pred_args = ['--model-dir', str(MODEL_DIR), '--image', str(example_image), '--topk', '3', '--out', str(REPORTS_RUN_DIR/'predict_example.json')]
# rc = run_script('scripts/predict.py', pred_args)
# if rc != 0:
#     raise RuntimeError(f'Predição falhou (rc={rc}).')
print('Predição: descomente o bloco e ajuste paths para executar.')


## Exibir artefatos gerados (figuras e métricas)

Este bloco procura arquivos em `reports/` para registrar evidências no notebook.


In [ ]:
from IPython.display import Image, display
import json

print('Reports dir:', REPORTS_RUN_DIR)
imgs = sorted(REPORTS_RUN_DIR.rglob('*.png'))
print('Imagens:', len(imgs))
for p in imgs[:10]:
    print(' -', p)
    display(Image(filename=str(p)))

jsons = sorted(REPORTS_RUN_DIR.rglob('*.json'))
print('JSONs:', len(jsons))
if jsons:
    mf = jsons[-1]
    print('Último JSON:', mf)
    data = json.loads(mf.read_text(encoding='utf-8'))
    data


Reports dir (target): D:\workspace-python\xray-bone-fracture-classifier\reports\run
Imagens encontradas:

Nenhum arquivo .json de métricas encontrado em reports/<run_id>/ (ok se seu script não gera JSON).


## (Opcional) Comparar execuções

Útil para comparar baseline vs transfer learning, ou diferentes hiperparâmetros.


In [ ]:
# Descomente se você tiver múltiplas execuções em runs/
# run_script(
#     "scripts/compare_runs.py",
#     ["--runs-dir", str(RUNS_DIR), "--output-dir", str(REPORTS_DIR)],
# )
print("Compare runs: bloco opcional.")


Compare runs: bloco opcional.


## (Opcional) Inferência e Grad-CAM

Você pode:
- Reutilizar `scripts/predict.py` para inferência em poucas imagens
- Gerar overlays / mapas Grad-CAM (se o script suportar) e salvar em `reports/`

> Dica: se o Grad-CAM estiver integrado apenas ao Streamlit, você pode documentar aqui o procedimento e incluir as evidências exportadas.


In [ ]:
# Inferência via CLI (scripts/predict.py)
# usage: predict.py --model-dir ... [--image ...] [--input-dir ...] [--topk ...] [--out ...]

# example_image = DATA_PROCESSED_DIR / "test" / "<classe>" / "example.png"

# run_script(
#     "scripts/predict.py",
#     [
#         "--model-dir", str(MODEL_DIR),
#         "--image", str(example_image),
#         "--topk", "3",
#         "--out", str(REPORTS_RUN_DIR / "predict_example.json"),
#     ],
# )

print("Inferência: bloco opcional (ajuste paths conforme seu dataset).")


Inferência: bloco opcional (ajuste paths conforme seu dataset).


## Discussão e conclusões

Inclua aqui (curto e objetivo):
- Melhor configuração e principais evidências (métricas + matriz de confusão)
- Erros típicos e hipóteses (ruído, classes parecidas, qualidade de imagem)
- Limitações e próximos passos
